# State-based History-aware Artificial Reinforcement Interpretable Kernel" (Sharik)
## Open AI Gym - Atari - Breakout - Experiential Learning - Exploring Extended Hyper-Parameters  

- $mc$ --motivated_curiosity, type=float, default=0.0 - Motivated (by surprise) curiosity
  - $c_t$ chance to perform random action ("curiosity"), dependant on "surprizingness" $z_t = norm\_sim(s_t,s'_t)$, as $c_t = mc * z_t$
- $cc$ --constant curiosity (inverse to greediness) - chance to perform random action regardless of the "surprizingness"  
- $er$ --expectedness_reward, type=float, default=0.0 - Expectedness reward
  - $g_t$ - extra positive feedback for predictiviness $1 - z_t$, as $g_t = er * (1-z_t)$, where $er$ can be considered as an element of agent's personal profile $x$
- $sr$ --state_reward, type=int, default=1 - Place reward in state (1) or not (0)
  
So far, none of them works - for seeds 2, 3, and 41 the best results are still for $er=0, mc=0, cc=0, sr=1$


In [1]:
import os, sys
import pickle
import pandas as pd
import queue
import matplotlib.pyplot as plt
import random
import math
import numpy as np

## Experiment with Motivated (Conditioned) Curiosity and Expectedness (Predictiveness) Reward based on Surprisingness (anti-Predictiveness)

In [14]:
# er, mc: 1M

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc0.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=0.5 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er05mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=1.0 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er10mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=0.5 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=1.0 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc10.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=0.5 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er05mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=1.0 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er10mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=0.5 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=1.0 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc10.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=0.5 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er05mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=1.0 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er10mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=0.5 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=1.0 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc10.txt

#=>

#er0mc0 => 8.83
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:08.072909"; time="0:19:32.297677"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:02.481580"; time="0:19:42.842152"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.756226"; time="0:19:42.002109"

#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:08.072909"; time="0:19:32.297677"
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc05.txt:score_avg=4.3; steps_tot=1080000; steps_avg=1118.0; lives_avg=0.0; lapse_avg="0:00:00.570908"; time="0:08:58.610062"
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc10.txt:score_avg=5.5; steps_tot=1080000; steps_avg=1399.0; lives_avg=0.0; lapse_avg="0:00:00.194261"; time="0:22:43.029088"
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er05mc0.txt:score_avg=0.2; steps_tot=1080000; steps_avg=563.4; lives_avg=0.0; lapse_avg="0:00:00.242842"; time="0:10:05.387658"
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er10mc0.txt:score_avg=1.0; steps_tot=1080000; steps_avg=847.1; lives_avg=0.0; lapse_avg="0:00:00.069858"; time="0:10:36.342916"

#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:02.481580"; time="0:19:42.842152"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc05.txt:score_avg=3.6; steps_tot=1080000; steps_avg=1609.5; lives_avg=0.1; lapse_avg="0:00:01.505553"; time="0:08:47.056933"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc10.txt:score_avg=3.5; steps_tot=1080000; steps_avg=1005.6; lives_avg=0.0; lapse_avg="0:00:00.057235"; time="0:22:46.613459"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er05mc0.txt:score_avg=3.0; steps_tot=1080000; steps_avg=915.3; lives_avg=0.0; lapse_avg="0:00:00.343216"; time="0:10:05.306984"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er10mc0.txt:score_avg=3.0; steps_tot=1080000; steps_avg=1013.1; lives_avg=0.0; lapse_avg="0:00:00.390394"; time="0:10:37.310238"

#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.756226"; time="0:19:42.002109"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc05.txt:score_avg=6.0; steps_tot=1080000; steps_avg=1438.1; lives_avg=0.0; lapse_avg="0:00:00.030599"; time="0:08:51.614975"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc10.txt:score_avg=4.1; steps_tot=1080000; steps_avg=1193.4; lives_avg=0.0; lapse_avg="0:00:00.559716"; time="0:22:47.393618"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er05mc0.txt:score_avg=1.0; steps_tot=1080000; steps_avg=726.3; lives_avg=0.0; lapse_avg="0:00:00.127887"; time="0:10:06.467727"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er10mc0.txt:score_avg=1.0; steps_tot=1080000; steps_avg=716.2; lives_avg=0.0; lapse_avg="0:00:00.014154"; time="0:10:38.636689"


# er, mc: 5M
# python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)"         > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er05mc0.txt
# python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -er=0.5 > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er05mc0.txt
# python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -er=1.0 > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er10mc0.txt
# python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -mc=0.5 > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er0mc05.txt
# python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -mc=1.0 > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er0mc10.txt


#=>

#er0mc0 => THE BEST
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=22.2; steps_tot=5400000; steps_avg=4048.0; lives_avg=0.2; lapse_avg="0:00:01.626634"; time="0:50:52.562686"
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er05mc0.txt:score_avg=0.2; steps_tot=5400000; steps_avg=541.8; lives_avg=0.0; lapse_avg="0:00:00.374476"; time="2:35:49.042470"
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er0mc05.txt:score_avg=6.8; steps_tot=5400000; steps_avg=1419.9; lives_avg=0.0; lapse_avg="0:00:00.229527"; time="0:53:03.293327"
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er0mc10.txt:score_avg=7.5; steps_tot=5400000; steps_avg=1815.7; lives_avg=0.0; lapse_avg="0:00:00.075259"; time="0:47:57.899908"
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er10mc0.txt:score_avg=1.0; steps_tot=5400000; steps_avg=712.3; lives_avg=0.0; lapse_avg="0:00:00.388237"; time="0:45:08.367108"

#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=14.0; steps_tot=5400000; steps_avg=2880.0; lives_avg=0.1; lapse_avg="0:00:05.784216"; time="0:51:09.915017"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er05mc0.txt:score_avg=3.0; steps_tot=5400000; steps_avg=906.2; lives_avg=0.0; lapse_avg="0:00:00.176974"; time="2:33:20.356969"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er0mc05.txt:score_avg=5.9; steps_tot=5400000; steps_avg=2222.2; lives_avg=0.1; lapse_avg="0:00:07.573966"; time="0:52:39.433090"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er0mc10.txt:score_avg=5.3; steps_tot=5400000; steps_avg=1251.2; lives_avg=0.0; lapse_avg="0:00:00.120064"; time="0:48:43.709109"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er10mc0.txt:score_avg=3.0; steps_tot=5400000; steps_avg=989.7; lives_avg=0.0; lapse_avg="0:00:00.472525"; time="0:44:13.115852"

#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=20.4; steps_tot=5400000; steps_avg=2825.7; lives_avg=0.0; lapse_avg="0:00:00.280927"; time="0:51:24.698343"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er05mc0.txt:score_avg=1.0; steps_tot=5400000; steps_avg=688.6; lives_avg=0.0; lapse_avg="0:00:00.184670"; time="2:32:34.406254"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er0mc05.txt:score_avg=8.7; steps_tot=5400000; steps_avg=1747.6; lives_avg=0.0; lapse_avg="0:00:01.743051"; time="0:53:42.183637"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er0mc10.txt:score_avg=7.1; steps_tot=5400000; steps_avg=1563.0; lives_avg=0.0; lapse_avg="0:00:00.115268"; time="0:48:22.910578"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er10mc0.txt:score_avg=1.0; steps_tot=5400000; steps_avg=688.6; lives_avg=0.0; lapse_avg="0:00:00.104093"; time="0:44:41.996405"



## Experiment with State Reward (Reward in State)  

In [9]:
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sr=1 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0sr1.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sr=1 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0sr1.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41  -mg=5000000 -mt=1080000 -sr=1 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0sr1.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sr=0 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0sr0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sr=0 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0sr0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sr=0 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0sr0.txt

# sr1 => 8.83 (best)
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0sr1.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:08.468077"; time="0:11:17.523051"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0sr1.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:02.342758"; time="0:11:31.564646"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0sr1.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.757189"; time="0:14:26.142090"

# sr0 => 6.5
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0sr0.txt:score_avg=4.1; steps_tot=1080000; steps_avg=1483.5; lives_avg=0.1; lapse_avg="0:00:00.875952"; time="0:11:35.934361"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0sr0.txt:score_avg=9.5; steps_tot=1080000; steps_avg=2080.9; lives_avg=0.1; lapse_avg="0:00:01.080624"; time="0:11:32.168979"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0sr0.txt:score_avg=6.0; steps_tot=1080000; steps_avg=2926.8; lives_avg=0.2; lapse_avg="0:00:19.326955"; time="0:14:25.349930"

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=5400000 -sr=1 > sim_test4/exp5400000_s2ss10lm2sc2cs2tu0sr1.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=5400000 -sr=1 > sim_test4/exp5400000_s3ss10lm2sc2cs2tu0sr1.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41  -mg=5000000 -mt=5400000 -sr=1 > sim_test4/exp5400000_s41ss10lm2sc2cs2tu0sr1.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=5400000 -sr=0 > sim_test4/exp5400000_s2ss10lm2sc2cs2tu0sr0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=5400000 -sr=0 > sim_test4/exp5400000_s3ss10lm2sc2cs2tu0sr0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=5400000 -sr=0 > sim_test4/exp5400000_s41ss10lm2sc2cs2tu0sr0.txt

# sr1 => 18.8
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0sr1.txt:score_avg=22.2; steps_tot=5400000; steps_avg=4048.0; lives_avg=0.2; lapse_avg="0:00:01.262753"; time="1:03:12.349516"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0sr1.txt:score_avg=14.0; steps_tot=5400000; steps_avg=2880.0; lives_avg=0.1; lapse_avg="0:00:04.120212"; time="1:03:21.890194"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0sr1.txt:score_avg=20.4; steps_tot=5400000; steps_avg=2825.7; lives_avg=0.0; lapse_avg="0:00:00.196652"; time="1:03:20.204869"

# sr0 => 12.47
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0sr0.txt:score_avg=6.9; steps_tot=5400000; steps_avg=1706.2; lives_avg=0.0; lapse_avg="0:00:00.046152"; time="0:59:36.553148"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0sr0.txt:score_avg=19.5; steps_tot=5400000; steps_avg=3184.0; lives_avg=0.1; lapse_avg="0:00:00.625007"; time="0:59:15.546036"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0sr0.txt:score_avg=11.0; steps_tot=5400000; steps_avg=3688.5; lives_avg=0.3; lapse_avg="0:00:00.413047"; time="0:59:01.144719"



In [15]:
#TODO???
# exclude/encode action: 1M, 5M
